# U.S. Weather Trends Analysis

This notebook loads the processed weather data from SQLite and shows it as a Pandas DataFrame.

In [ ]:
import sqlite3
import pandas as pd

db_path = 'data/processed/weather.db'
conn = sqlite3.connect(db_path)

current_df = pd.read_sql_query('SELECT * FROM weather_current', conn)
daily_df = pd.read_sql_query(
    '''
    SELECT city, date, temperature_2m_max_c, temperature_2m_min_c, precipitation_sum_mm, relative_humidity_2m_mean_pct, wind_speed_10m_max_kmh
    FROM weather_daily
    ORDER BY date DESC
    LIMIT 10
    '''
    , conn
)

print('CURRENT TABLE')
print(current_df)
print('\nDAILY TABLE HEAD')
print(daily_df)

conn.close()

## Weather trend analysis

This notebook loads the daily weather records from SQLite, calculates a few key indicators, and plots the recent climate signal for New York.

We will look at:
- daily average temperature
- 7-day rolling average temperature
- precipitation totals
- a quick summary of the latest observations


In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Connect to the processed weather database
conn = sqlite3.connect('data/processed/weather.db')

daily_df = pd.read_sql_query(
    '''
    SELECT
        city,
        date,
        temperature_2m_max_c,
        temperature_2m_min_c,
        precipitation_sum_mm,
        relative_humidity_2m_mean_pct,
        wind_speed_10m_max_kmh
    FROM weather_daily
    ORDER BY date ASC
    ''',
    conn,
)

conn.close()

daily_df['date'] = pd.to_datetime(daily_df['date'])
daily_df['temp_avg_c'] = (daily_df['temperature_2m_max_c'] + daily_df['temperature_2m_min_c']) / 2
daily_df['rolling_temp_7d'] = daily_df['temp_avg_c'].rolling(7, min_periods=1).mean()
daily_df['rolling_precip_7d'] = daily_df['precipitation_sum_mm'].rolling(7, min_periods=1).mean()

print(daily_df.head())
print('\nSummary statistics')
print(
    daily_df[['temperature_2m_max_c', 'temperature_2m_min_c', 'precipitation_sum_mm', 'relative_humidity_2m_mean_pct']]
    .describe()
    .round(2)
)

latest = daily_df.iloc[-1]
print('\nLatest record:')
print({
    'date': latest['date'].date(),
    'avg_temp_c': round(latest['temp_avg_c'], 2),
    'max_temp_c': round(latest['temperature_2m_max_c'], 2),
    'precipitation_mm': round(latest['precipitation_sum_mm'], 2),
})


In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid')

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

sns.lineplot(data=daily_df, x='date', y='temp_avg_c', ax=axes[0], color='#1f77b4', linewidth=2)
sns.lineplot(data=daily_df, x='date', y='rolling_temp_7d', ax=axes[0], color='#d62728', linewidth=2)
axes[0].set_title('Daily average temperature and 7-day rolling average')
axes[0].set_ylabel('Temperature (°C)')
axes[0].legend(['Daily average', '7-day rolling avg'])

sns.lineplot(data=daily_df, x='date', y='precipitation_sum_mm', ax=axes[1], color='#2ca02c', linewidth=2)
sns.lineplot(data=daily_df, x='date', y='rolling_precip_7d', ax=axes[1], color='#9467bd', linewidth=2)
axes[1].set_title('Daily precipitation and 7-day rolling average')
axes[1].set_ylabel('Precipitation (mm)')
axes[1].legend(['Daily total', '7-day rolling avg'])

fig.autofmt_xdate()
plt.tight_layout()
plt.show()
